In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision.datasets import FashionMNIST
from torchvision import transforms
from torch.utils.data import DataLoader

import numpy as np

from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [2]:
torch.manual_seed(123)
np.random.seed(123)

In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])
train_dataset = FashionMNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = FashionMNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

100%|███████████████████████████████████████████████████████████| 26.4M/26.4M [00:05<00:00, 4.95MB/s]
100%|████████████████████████████████████████████████████████████| 29.5k/29.5k [00:00<00:00, 147kB/s]
100%|███████████████████████████████████████████████████████████| 4.42M/4.42M [00:02<00:00, 1.92MB/s]
100%|███████████████████████████████████████████████████████████████████| 5.15k/5.15k [00:00<?, ?B/s]


In [ ]:
batch_size = 64

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [ ]:
class MLP(nn.Module):

    def __init__(self):
        super(MLP, self).__init__()

        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
model = MLP()

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
epochs = 10

for epoch in range(epochs):

    running_loss = 0

    for images, labels in train_loader:

        outputs = model(images)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss:.4f}")

In [ ]:
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():

    for images, labels in test_loader:

        outputs = model(images)

        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())

In [ ]:
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():

    for images, labels in test_loader:

        outputs = model(images)

        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())

In [ ]:
conf_matrix = confusion_matrix(all_labels, all_preds)

print("Confusion Matrix:")
print(conf_matrix)

From the confusion matrix, the model struggles most with distinguishing Shirt and T-shirt/top, which is expected because these clothing items have very similar visual features.

The model performs better on visually distinctive items such as Sneakers, Bags, and Ankle boots, which have unique shapes compared to other categories.

In [ ]:
precision = precision_score(all_labels, all_preds, average='macro')
recall = recall_score(all_labels, all_preds, average='macro')
f1 = f1_score(all_labels, all_preds, average='macro')

print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Hyperparameter chosen: Learning Rate

The learning rate determines how large the parameter updates are during training.
A value that is too high can cause unstable training, while a value that is too low can slow convergence.

I selected a learning rate of 0.001 using the Adam optimizer because it provides stable convergence and is commonly effective for image classification tasks.